In [1]:
import pandas as pd
import json

def parse_comments_json(filepath='comments.json'):
    comments_data = []

    # Open the file and read line by line (NDJSON format)
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line: continue

            try:
                # Parse JSON line
                obj = json.loads(line)

                # Extract fields based on your file's specific keys
                comments_data.append({
                    'username': obj.get('author'),
                    'timestamp_text': obj.get('time'),
                    'comment_text': obj.get('text')
                })
            except json.JSONDecodeError:
                continue

    return pd.DataFrame(comments_data)

# Load the data
comments_df = parse_comments_json('comments.json')
print(f"Loaded {len(comments_df)} comments.")
print(comments_df.head())

Loaded 58 comments.
              username timestamp_text  \
0  @UntamedNatureHub83    3 weeks ago   
1        @DailyShelter    1 month ago   
2   @FlightVectorLogic    1 month ago   
3      @NineDaysinEden    1 month ago   
4    @leonardosolis616    1 month ago   

                                        comment_text  
0  What an absolutely beautiful and emotional sho...  
1  It’s a nice wildlife video. Greetings from Bot...  
2  It's a good thing the team's aircraft served a...  
3  Beautiful footage — it’s incredible to see the...  
4  Hey BBC Earth channel, upload more videos of T...  


In [5]:
import re
import pandas as pd

def structure_captions_from_vtt(filepath='captions.vtt'):
    captions = []

    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()

            # Filter out metadata, timestamps (-->), and empty lines as per Lab 2 Manual
            if '-->' not in line and line and not line.isdigit() and 'WEBVTT' not in line and 'Kind:' not in line and 'Language:' not in line:
                # Remove HTML-like tags (e.g., <c>) often found in VTT
                clean_line = re.sub(r'<[^>]+>', '', line)
                captions.append(clean_line)

    # Join all lines into one text block and split by sentence delimiters
    full_text = ' '.join(captions)
    # Split by sentence endings (. ! ?)
    sentences = re.split(r'(?<=[.!?]) +', full_text)

    return pd.DataFrame(sentences, columns=['caption_sentence'])

# Load the captions
captions_df = structure_captions_from_vtt('captions.vtt')
print(f"Loaded {len(captions_df)} caption sentences.")
print(captions_df.head())

Loaded 41 caption sentences.
                                    caption_sentence
0  [Music] The morning after the heavy rains, the...
1                             zebra start to arrive.
2                             zebra start to arrive.
3  Janet's family is the first to be seen Janet's...
4  She was the first zebra ever recorded to She w...


In [6]:
import nltk
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


True

In [8]:
import re
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import nltk
nltk.download('punkt_tab', quiet=True)

# Initialize tools
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text_pipeline(text):
    if not isinstance(text, str): return []

    # 1. Normalization: Lowercase and remove non-letters
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)

    # 2. Tokenization: Split into words
    tokens = word_tokenize(text)

    # 3. Stopword Removal & Filtering Short Words
    filtered_tokens = [w for w in tokens if w not in stop_words and len(w) > 2]

    # 4. Lemmatization: Convert to base form
    final_tokens = [lemmatizer.lemmatize(w) for w in filtered_tokens]

    return final_tokens

# Apply pipeline to DataFrames
comments_df['cleaned_tokens'] = comments_df['comment_text'].apply(clean_text_pipeline)
captions_df['cleaned_tokens'] = captions_df['caption_sentence'].apply(clean_text_pipeline)

# Display results
print("Cleaned Comments Sample:")
print(comments_df[['comment_text', 'cleaned_tokens']].head())

Cleaned Comments Sample:
                                        comment_text  \
0  What an absolutely beautiful and emotional sho...   
1  It’s a nice wildlife video. Greetings from Bot...   
2  It's a good thing the team's aircraft served a...   
3  Beautiful footage — it’s incredible to see the...   
4  Hey BBC Earth channel, upload more videos of T...   

                                      cleaned_tokens  
0  [absolutely, beautiful, emotional, short, docu...  
1        [nice, wildlife, video, greeting, botswana]  
2  [good, thing, team, aircraft, served, purpose,...  
3  [beautiful, footage, incredible, see, nxai, pa...  
4  [hey, bbc, earth, channel, upload, video, tige...  


In [9]:
# Save to CSV without the index as instructed
comments_df.to_csv('cleaned_comments.csv', index=False)
captions_df.to_csv('cleaned_captions.csv', index=False)

print("Files exported successfully: cleaned_comments.csv, cleaned_captions.csv")

Files exported successfully: cleaned_comments.csv, cleaned_captions.csv
